# Day 9/90 - Machine Learning Concepts

We will use one simple housing dataset for most of the lesson:


The main goal is to understand how machine-learning problems are organized.

```text
Features and Labels

Supervised Learning
├── A. Regression
└── B. Classification

Unsupervised Learning
└── A. Clustering

Self-Supervised Learning

Ranking

Baselines
```


##Why do we need to learn these ML concepts?

You can build simple AI apps by calling an existing model, but to build systems that are actually reliable, you need to understand what the model is doing underneath. These ML concepts help you choose the right kind of model for a task, understand what data it needs, interpret its outputs, compare approaches, and debug problems when results are poor. They also make later topics like embeddings, RAG, reranking, model evaluation, and agents much easier to reason about, because instead of treating the model as a black box, you understand how it learns, what it predicts, and how to tell whether it is working well.

# 1. Load the Housing Dataset

We will use the **Ames Housing dataset**.

To keep Day 9 simple, we will use only a few easy-to-understand columns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, accuracy_score

# Load the Ames Housing dataset.
housing = fetch_openml(
    name="house_prices",
    as_frame=True,
    parser="auto"
)

raw = housing.frame.copy()

# Keep only simple columns.
df = raw[
    [
        "GrLivArea",      # living area in square feet
        "BedroomAbvGr",   # bedrooms
        "FullBath",       # full bathrooms
        "GarageCars",     # garage capacity
        "YearBuilt",      # year built
        "SalePrice"       # sale price in dollars
    ]
].copy()

# Rename them to beginner-friendly names.
df = df.rename(columns={
    "GrLivArea": "living_area_sqft",
    "BedroomAbvGr": "bedrooms",
    "FullBath": "bathrooms",
    "GarageCars": "garage_spaces",
    "YearBuilt": "year_built",
    "SalePrice": "sale_price"
})

# Remove rows with missing values in the selected columns.
df = df.dropna().reset_index(drop=True)

df.head()

# 2. Features and Labels

### Features

**Features are the input values given to the model.**

Housing example:

```text
living area
bedrooms
bathrooms
garage spaces
year built
```

### Label / target

The **label or target** is the answer we want the model to learn to predict.

Housing example:

```text
sale price
```

So:

```text
house features
      ↓
model
      ↓
sale price
```

### Where this shows up in an AI system

Suppose you want to predict whether a support ticket should go to the billing team.

Your features could be information extracted from the message:

```text
words / text representation
customer plan
previous issue type
account status
```

Your label could be:

```text
billing_issue
```

During training:

```text
features
+
known label
↓
model learns the relationship
```

In ML code you will commonly see:

```text
X -> features
y -> target / label
```

The first question in a supervised ML problem is always:

> **What information goes in, and what answer should come out?**

In [ ]:
# X = features given to the model.
X = df[
    [
        "living_area_sqft",
        "bedrooms",
        "bathrooms",
        "garage_spaces",
        "year_built"
    ]
]

# y = target we want to predict.
y_price = df["sale_price"]

display(X.head())
display(y_price.head())

# 3. Supervised Learning

**Supervised learning means the training data already contains the correct answer.**

Housing example:

```text
INPUT
living area = 1710 sq ft
bedrooms = 3
bathrooms = 2

CORRECT ANSWER
sale price = $208,500
```

The model sees many examples like this:

```text
features + correct answer
          ↓
        learns pattern
          ↓
predicts answer for new data
```

The two common supervised-learning problems we need here are:

```text
A. Regression
-> predict a number

B. Classification
-> predict a category
```

### Where this shows up in an AI system

Imagine you are building a support AI system.

You already have old support messages with the correct intent attached:

```text
"I was charged twice"
-> billing_issue

"I cannot log in"
-> login_issue

"How do I cancel?"
-> cancellation
```

That is supervised learning because the model is learning from examples where the correct answer is already known.

```text
user message
     ↓
trained model
     ↓
predicted intent
```

The predicted intent could then be used to route the request to the correct workflow or agent.

## A. Regression - Predict a Number

**Regression is a supervised-learning problem where the output is a number.**

### Housing example

Input:

```text
living area   = 1710 sq ft
bedrooms      = 3
bathrooms     = 2
garage spaces = 2
year built    = 2003
```

Output:

```text
predicted sale price = $210,000
```

Because the model returns a numerical value, this is regression.

### Where this shows up in an AI system

Suppose you want your AI application to estimate how long a request will take.

Input:

```text
prompt length
number of tool calls
model used
context size
```

Model output:

```text
predicted latency = 4.8 seconds
```

That is regression because the answer is a number.

Other examples:

```text
predict cost
predict delivery time
predict demand
predict model latency
predict expected revenue
```

Use regression when the question is:

> **How much? How long? How many? What numerical value?**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_price,
    test_size=0.2,
    random_state=42
)

# Simple regression model.
reg_model = LinearRegression()

# Learn from training examples.
reg_model.fit(X_train, y_train)

# Predict prices for unseen homes.
price_predictions = reg_model.predict(X_test)

print("Predicted prices:")
print(np.round(price_predictions[:5], 0))

print("\nActual prices:")
print(y_test.to_numpy()[:5])

In [ ]:
# Mean Absolute Error = average absolute prediction error.
reg_mae = mean_absolute_error(
    y_test,
    price_predictions
)

print("Mean Absolute Error: $", round(reg_mae, 2))

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    y_test,
    price_predictions,
    alpha=0.5
)

low = min(y_test.min(), price_predictions.min())
high = max(y_test.max(), price_predictions.max())

plt.plot(
    [low, high],
    [low, high],
    linestyle="--"
)

plt.xlabel("Actual sale price")
plt.ylabel("Predicted sale price")
plt.title("Regression: Actual vs Predicted Price")
plt.grid(alpha=0.3)
plt.show()

### What this graph shows

- **x-axis:** real sale price
- **y-axis:** predicted sale price
- dots closer to the diagonal = better predictions

Main idea:

```text
output is a number
-> regression
```

## B. Classification - Predict a Category

**Classification is a supervised-learning problem where the output is a category.**

### Housing example

Instead of predicting the exact price, ask:

```text
Will this house sell for more than $200,000?
```

Possible outputs:

```text
0 -> $200,000 or below
1 -> above $200,000
```

Example:

```text
house features
      ↓
classification model
      ↓
above $200k
```

Because the output is a category, this is classification.

### Where this shows up in an AI system

Suppose you build an AI support system.

User says:

```text
"I was charged twice this month"
```

A classifier could return:

```text
billing_issue
```

Then your application can route the request:

```text
user message
      ↓
intent classifier
      ↓
billing_issue
      ↓
billing agent / billing workflow
```

Other classification examples:

```text
email -> spam / not spam
transaction -> fraud / not fraud
document -> invoice / contract / resume
message -> billing / technical / account
content -> safe / unsafe
```

Use classification when the question is:

> **Which category does this belong to?**

In [ ]:
# Create a category from the sale price.
df["above_200k"] = (
    df["sale_price"] > 200_000
).astype(int)

display(
    df[
        ["sale_price", "above_200k"]
    ].head(10)
)

In [ ]:
X_class = X
y_class = df["above_200k"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

class_model = LogisticRegression(
    max_iter=1000
)

class_model.fit(
    X_train_c,
    y_train_c
)

class_predictions = class_model.predict(
    X_test_c
)

class_accuracy = accuracy_score(
    y_test_c,
    class_predictions
)

print("Classification accuracy:", round(class_accuracy, 3))

The difference:

```text
Predict exact price
-> Regression

Predict above / below $200,000
-> Classification
```

Same dataset.

Different question.

# 4. Unsupervised Learning

**Unsupervised learning means we have data, but we do not give the algorithm a correct target to predict.**

Housing example:

```text
living area
bedrooms
bathrooms
garage spaces
year built
```

But we do not have labels such as:

```text
small_home
family_home
luxury_home
```

The algorithm tries to discover structure from the data itself.

```text
unlabeled data
      ↓
find patterns / similarities
      ↓
groups or structure
```

### What comes under unsupervised learning?

Common tasks include:

```text
Clustering
-> discover groups

Dimensionality reduction
-> compress many features into fewer dimensions

Anomaly detection
-> find unusual examples
```

For Day 9, we only need to understand **clustering**.

### Where this shows up in an AI system

Imagine you have thousands of user conversations but no intent labels.

You could represent each conversation numerically and look for natural groups.

You might discover:

```text
Group 1 -> billing-like conversations
Group 2 -> login problems
Group 3 -> onboarding questions
```

You did not tell the algorithm those categories beforehand.

That is the key difference from supervised learning.

## A. Clustering - Discover Groups

**Clustering groups similar examples together without being told what the groups should be.**

### Housing example

Suppose we have:

```text
House A -> 900 sq ft, 2 beds, 1 bath
House B -> 1050 sq ft, 2 beds, 1 bath
House C -> 2800 sq ft, 4 beds, 3 baths
House D -> 3100 sq ft, 5 beds, 3 baths
```

A clustering algorithm may discover:

```text
Cluster 0
-> smaller homes

Cluster 1
-> medium homes

Cluster 2
-> larger homes
```

The names are added by us after inspecting the clusters. The algorithm only finds groups based on similarity.

### Where this shows up in an AI system

Suppose you have embeddings for 10,000 user conversations.

Clustering can help you discover groups such as:

```text
similar billing questions
similar technical issues
similar onboarding questions
```

Flow:

```text
conversation embeddings
        ↓
clustering
        ↓
groups of similar conversations
        ↓
inspect each group
        ↓
understand common user needs
```

Possible uses:

```text
discover user segments
group similar documents
explore embedding spaces
find repeated support problems
group similar products
```

Use clustering when you want to **discover groups before you already know what the groups are**.

In [ ]:
cluster_features = df[
    [
        "living_area_sqft",
        "bedrooms",
        "bathrooms"
    ]
]

# Scale the values because square feet is much larger
# numerically than bedrooms or bathrooms.
scaler = StandardScaler()

cluster_scaled = scaler.fit_transform(
    cluster_features
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(
    cluster_scaled
)

display(
    df[
        [
            "living_area_sqft",
            "bedrooms",
            "bathrooms",
            "cluster"
        ]
    ].head()
)

### What did the clusters actually find?

Instead of only seeing cluster numbers such as `0`, `1`, and `2`, let's calculate the average house characteristics inside each cluster.

That makes the groups easier to interpret.

In [ ]:
cluster_summary = (
    df.groupby("cluster")[
        [
            "living_area_sqft",
            "bedrooms",
            "bathrooms",
            "sale_price"
        ]
    ]
    .mean()
    .round(1)
)

cluster_summary

The cluster number itself has no built-in meaning.

You inspect the averages and interpret the groups.

For example, one cluster may contain homes that are generally:

```text
smaller
fewer bedrooms
lower sale price
```

while another may contain:

```text
larger
more bedrooms/bathrooms
higher sale price
```

That is the practical purpose of clustering: **discover groups first, then study what makes those groups different.**

### Why did we scale?

Example:

```text
living area = 1800
bedrooms   = 3
bathrooms  = 2
```

The numbers use very different ranges.

KMeans uses numerical distance, so we scale them first so one feature does not dominate just because its numbers are larger.

You will study preprocessing more deeply later.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["living_area_sqft"],
    df["bedrooms"],
    c=df["cluster"],
    alpha=0.5
)

plt.xlabel("Living area (sq ft)")
plt.ylabel("Bedrooms")
plt.title("Housing Clusters")
plt.grid(alpha=0.3)
plt.show()

### What this graph shows

- each dot = one home
- x-axis = living area
- y-axis = number of bedrooms
- color = cluster found by the algorithm

No correct cluster labels were provided.

# 6. Ranking - Decide What Comes First

**Ranking means ordering several items from most useful/relevant to least useful/relevant.**

### Housing example

Suppose predicted prices are:

```text
House A -> $510,000
House B -> $340,000
House C -> $720,000
House D -> $450,000
```

Ranking gives:

```text
1. House C -> $720,000
2. House A -> $510,000
3. House D -> $450,000
4. House B -> $340,000
```

### Where this shows up in an AI system

A very common example is **RAG**.

User asks:

```text
"How do I reset my password?"
```

The retrieval system may find 20 document chunks.

Each chunk gets a relevance score:

```text
Password Reset Guide      -> 0.94
Account Recovery          -> 0.81
Login Troubleshooting     -> 0.73
Billing FAQ               -> 0.11
```

Ranking sorts them:

```text
1. Password Reset Guide
2. Account Recovery
3. Login Troubleshooting
4. Billing FAQ
```

Then the top results can be sent to the LLM as context.

```text
query
  ↓
retrieve candidates
  ↓
score / rank them
  ↓
top documents
  ↓
LLM
```

Ranking is also used in:

```text
search
recommendations
feeds
ads
job matching
```

Use ranking when the question is:

> **Which result should come first?**

In [ ]:
# Predict a price score for every home.
all_price_scores = reg_model.predict(X)

ranked_homes = df[
    [
        "living_area_sqft",
        "bedrooms",
        "bathrooms",
        "garage_spaces",
        "sale_price"
    ]
].copy()

ranked_homes["predicted_price"] = all_price_scores

ranked_homes = ranked_homes.sort_values(
    "predicted_price",
    ascending=False
)

ranked_homes.head(10)

The difference:

```text
Classification -> Which category?
Regression     -> What number?
Ranking        -> Which item comes first?
```

# 7. Baselines - Is the Model Actually Better?

**A baseline is a simple starting solution used as a reference point.**

Without a baseline, a model score has very little meaning.

### Housing regression example

Simple baseline:

```text
ignore the house features
always predict the average sale price
```

Then compare:

```text
baseline error = $55,000
model error    = $25,000
```

Now we know the trained model improved over the simple guess.

### Where this shows up in an AI system

Suppose you build a new AI agent.

Current system:

```text
one prompt + one tool call
success rate = 72%
```

New agent:

```text
planner + tools + retries + memory
success rate = 74%
```

The current simple system is your baseline.

Now you can ask:

```text
Did the extra complexity actually help?
```

The same idea applies to:

```text
new RAG pipeline
vs
simple keyword search

new reranker
vs
original retrieval order

new prompt
vs
current prompt

new model
vs
simpler model
```

A baseline gives you something concrete to beat.

> **If the new system does not beat a reasonable baseline, the extra complexity may not be worth it.**

In [ ]:
# Regression baseline.
average_price = y_train.mean()

baseline_price_predictions = np.full(
    len(y_test),
    average_price
)

baseline_reg_mae = mean_absolute_error(
    y_test,
    baseline_price_predictions
)

print("Baseline regression MAE: $", round(baseline_reg_mae, 2))
print("Model regression MAE   : $", round(reg_mae, 2))

In [ ]:
# Classification baseline.
majority_class = y_train_c.mode()[0]

baseline_class_predictions = np.full(
    len(y_test_c),
    majority_class
)

baseline_class_accuracy = accuracy_score(
    y_test_c,
    baseline_class_predictions
)

print("Baseline accuracy:", round(baseline_class_accuracy, 3))
print("Model accuracy   :", round(class_accuracy, 3))

How to read them:

```text
Accuracy
-> higher is better

MAE
-> lower is better
```

Main question:

> **Did the model beat the simple baseline?**

# 5. Self-Supervised Learning - Learn From Raw Data

**Self-supervised learning creates the training target from the raw data itself.**

This idea is clearer with text than housing data.

### Text example

Raw sentence:

```text
I love machine learning
```

We can automatically create:

```text
INPUT
I love machine

TARGET
learning
```

Nobody manually labeled the sentence.

The answer already existed inside the raw text.

Another example:

```text
AI systems learn from data
```

becomes:

```text
INPUT
AI systems learn from

TARGET
data
```

### Where this shows up in AI

This is one of the main ideas behind training foundation models.

For an LLM:

```text
raw text
   ↓
create next-token prediction examples
   ↓
model predicts next token
   ↓
compare prediction with actual token
   ↓
update model
```

Example:

```text
input:
"The user forgot their"

target:
"password"
```

The model can learn from huge amounts of text without humans manually labeling every sentence.

Self-supervised learning is also used with:

```text
images
audio
video
```

For Day 9, remember:

> **Supervised learning uses externally provided labels. Self-supervised learning creates the training signal from the raw data itself.**

In [ ]:
sentences = [
    "I love machine learning",
    "AI systems learn from data",
    "models predict useful outputs"
]

examples = []

for sentence in sentences:
    words = sentence.split()

    examples.append({
        "input": " ".join(words[:-1]),
        "target": words[-1]
    })

pd.DataFrame(examples)

The raw text itself produced the target.

That is the key self-supervised idea.

# 8. How the Concepts Fit Together

The housing dataset lets us see the main structure clearly:

```text
SUPERVISED LEARNING
correct target is known

├── Regression
│   -> predict exact sale price
│
└── Classification
    -> predict above / below $200,000


UNSUPERVISED LEARNING
no correct target is supplied

└── Clustering
    -> discover groups of similar homes


SELF-SUPERVISED LEARNING
target is created from raw data

-> tiny text example


RANKING
-> order items by a score


BASELINE
-> simple reference used to judge improvement
```

The important point is:

> **The type of machine-learning problem depends on what data you have and what output you want.**

# 9. Quick Reference

| Main concept | Subtype / task | Simple meaning | Example use |
|---|---|---|---|
| Features / labels | - | input information and target answer | almost every ML system |
| Supervised learning | Regression | predict a number | price, latency, demand |
| Supervised learning | Classification | predict a category | spam, fraud, intent routing |
| Unsupervised learning | Clustering | discover groups without labels | customers, documents, behavior |
| Self-supervised learning | - | create training targets from raw data | LLM pretraining |
| Ranking | - | order items by a score | search, RAG, recommendations |
| Baseline | - | simple comparison point | check whether a new system improved |

# 10. Practice

Using the housing dataset:

### 1.
Predict exact sale price.

### 2.
Predict whether a house is above $200,000.

### 3.
Remove the price target and group similar homes.

### 4.
Rank homes from highest predicted price to lowest.

### 5.
Create a simple regression baseline.

For each one, answer:

```text
What are the features?
What is the target?
Do labels exist?
What type of ML problem is this?
```

Then explain:

> Why is the raw-text example self-supervised?

**Practice, don't just read. Happy learning:)**